# Solving Simple Harmonic Motion (SHM) using DeepXDE (PINN)

This notebook demonstrates how to solve a second-order ordinary differential equation (ODE) describing Simple Harmonic Motion using Physics-Informed Neural Networks (PINNs) via the **DeepXDE** library.

> Normally, when we want to solve a differential equation, we either find a formula by hand (an *analytical* solution) or we use a numerical method like Runge-Kutta or finite differences, which steps forward in tiny increments of time.
>
> A **Physics-Informed Neural Network (PINN)** takes a completely different approach:
> - We build a neural network $x_\theta(t)$ (with trainable weights $\theta$) that takes time $t$ as input and outputs a *guess* for the displacement $x(t)$.
> - We don't show the network any "correct answers." Instead, we tell it the **physics law** the solution must obey (the ODE itself) and the **initial conditions**.
> - We then train the network so that, when we plug its output into the ODE, the equation is satisfied as closely as possible at many sample points in time — and so that it matches the initial conditions.
> - If training succeeds, the network has effectively *learned* a function that approximates the true solution $x(t)$, even though it never saw the analytical formula.


## Problem Setup
The governing equation for an ideal, undamped simple harmonic oscillator is given by:

$$\frac{d^2x}{dt^2} + \omega^2 x = 0$$

Where:
- $x(t)$ is the displacement at time $t$
- $\omega$ is the angular frequency (we will choose $\omega = 2$)

### Initial Conditions (IC):
- $x(0) = 1$ (Initial displacement)
- $\frac{dx}{dt}(0) = 0$ (Initial velocity)

### Analytical Solution:
$$x(t) = \cos(\omega t)$$

We can verify this solves the ODE by differentiating twice: $\frac{d^2}{dt^2}\cos(\omega t) = -\omega^2\cos(\omega t)$, so $\frac{d^2x}{dt^2} + \omega^2 x = -\omega^2\cos(\omega t) + \omega^2\cos(\omega t) = 0$. ✓ It also satisfies $x(0)=\cos(0)=1$ and $x'(0) = -\omega\sin(0) = 0$. ✓

We'll use this known formula later purely as a **ground truth** to check how accurate our trained PINN is — the network itself never gets to see this formula.

## Step 1: Imports and Setup

Before writing any physics, we load the libraries we'll need:

- **`deepxde`** (imported as `dde`) — the PINN framework that handles the neural network, the automatic differentiation needed to compute $\frac{d^2x}{dt^2}$, and the training loop.
- **`numpy`** — for basic numerical arrays and operations (e.g. generating the grid of time points we'll plot later).
- **`matplotlib.pyplot`** — for plotting our results at the end.

DeepXDE can run on top of several backends (TensorFlow, PyTorch, JAX, ...). We explicitly choose **PyTorch** by setting an environment variable *before* importing `deepxde`, since DeepXDE picks its backend at import time.

In [1]:
# Tell DeepXDE which backend (the underlying deep-learning engine) to use.
# This MUST be set before importing deepxde, since the backend is chosen at import time.
import os
os.environ["DDE_BACKEND"] = "pytorch"

import deepxde as dde      # the PINN library: builds the network, the physics loss, and trains
import numpy as np         # numerical arrays
import matplotlib.pyplot as plt  # plotting

Using backend: pytorch
Other supported backends: tensorflow.compat.v1, tensorflow, jax, paddle.
paddle supports more examples now and is recommended.


## Step 2: Define the ODE, Geometry, and Boundary/Initial Conditions

This is the heart of any PINN setup. DeepXDE needs to know three things:

1. **The domain** — what range of $t$ are we solving over? (here, $t \in [0, 5]$)
2. **The residual function** — a Python function that, given the network's prediction $x$ and the input $t$, returns *how far* the prediction is from satisfying the ODE. If the network were perfect, this residual would be exactly zero everywhere. During training, DeepXDE squares this residual and tries to drive it toward zero at many sampled points — this becomes part of the loss function.
3. **The initial/boundary conditions** — extra constraints (like $x(0)=1$) that the residual alone doesn't enforce, since a function that satisfies the ODE could still be shifted in phase or amplitude.

A quick vocabulary note: even though $t$ is "time," not a spatial coordinate, DeepXDE treats the 1D time domain the same way it would treat a 1D spatial domain, calling the conditions at its two timeline edges "boundary conditions." The condition at $t=0$ is what physicists would call an *initial condition* — DeepXDE just uses the more general PDE vocabulary for both.

In [2]:
omega = 2.0  # angular frequency of the oscillator (rad/s)


def shm_pde(t, x):
    """
    The ODE residual: d^2x/dt^2 + omega^2 * x.

    - `t` is the input (time) the network was evaluated at.
    - `x` is the network's *output* (its prediction for displacement) at those `t` values.
    - `dde.grad.hessian(x, t)` uses automatic differentiation (NOT finite differences!)
      to compute the exact second derivative d^2x/dt^2 of the network's output
      with respect to its input. This is one of the key tricks that makes PINNs work:
      the network itself is a differentiable function, so we can get exact derivatives
      "for free" via backpropagation.

    If x(t) perfectly solved the ODE, this function would return 0 everywhere.
    During training, DeepXDE samples many points t, evaluates this residual,
    and tries to push it toward zero (this becomes the "physics loss" term).
    """
    d2x_dt2 = dde.grad.hessian(x, t)
    return d2x_dt2 + omega**2 * x


def boundary_initial(t, on_boundary):
    """
    Tells DeepXDE which points count as "the initial condition point" (t = 0).

    `on_boundary` is True for points DeepXDE has already identified as being on
    the edge of the domain (for a 1D time domain [0, 5], that's t=0 and t=5).
    We additionally check `np.isclose(t[0], 0.0)` so that we only flag t=0
    (the *start* of time), not t=5, since our two conditions both apply at t=0.
    """
    return on_boundary and np.isclose(t[0], 0.0)


def true_solution(t):
    """
    The known analytical solution x(t) = cos(omega * t).

    NOTE: the network never sees this function during training.
    We only use it afterward to measure how accurate the PINN's prediction is,
    and DeepXDE also uses it internally to report a running error metric
    (see `metrics=["l2 relative error"]` later) purely for our own diagnostics.
    """
    return np.cos(omega * t)


# The computational domain: time runs from t=0 to t=5.
geom = dde.geometry.TimeDomain(0, 5)

# Initial condition #1: position. x(0) = 1.
# `lambda t: 1.0` just means "the target value at this boundary point is 1.0".
ic_pos = dde.icbc.IC(
    geom,
    lambda t: 1.0,
    boundary_initial,
)

# Initial condition #2: velocity. dx/dt(0) = 0.
# Unlike `IC` (which constrains the function value itself), `OperatorBC` lets us
# constrain an arbitrary *operator* applied to the network output — here, its first
# derivative (`dde.grad.jacobian(x, t)`, the velocity), which we force to equal 0
# at t=0 (the third argument to the lambda, "_", is unused but required by the API).
ic_vel = dde.icbc.OperatorBC(
    geom,
    lambda t, x, _: dde.grad.jacobian(x, t),
    boundary_initial,
)

# Bundle everything into a single "PDE problem" object that DeepXDE understands:
# the domain, the ODE residual to minimize, the list of IC/BC constraints,
# and how many sample points to use during training/testing.
data = dde.data.PDE(
    geom,
    shm_pde,
    [ic_pos, ic_vel],
    num_domain=200,    # number of random points inside [0, 5] where we enforce the ODE residual
    num_boundary=2,    # number of points on the domain's boundary (t=0 and t=5) used for IC/BC sampling
    solution=true_solution,  # optional: lets DeepXDE report the true error during training, for our own monitoring
    num_test=100,      # number of points used only to *evaluate* accuracy (not used in training itself)
)

## Step 3: Build and Train the Neural Network

Now we define the neural network itself and tell DeepXDE how to train it.

**The network architecture:** `[1, 32, 32, 32, 1]` means:
- **1** input neuron (time, $t$)
- three **hidden** layers of **32** neurons each
- **1** output neuron (the predicted displacement, $x$)

Each hidden neuron applies a weighted sum of its inputs followed by a nonlinear **activation function** — here, `tanh`. Nonlinearity is essential: without it, stacking layers would just compute another linear function of $t$, and a linear function can't represent something like $\cos(\omega t)$.

**The loss function** DeepXDE minimizes during training is a weighted sum of:
1. The ODE residual loss (how far `shm_pde` is from zero, averaged over the 200 domain points)
2. The initial-position loss (how far the network's $x(0)$ is from $1$)
3. The initial-velocity loss (how far the network's $x'(0)$ is from $0$)

Training adjusts the network's weights (via gradient descent, using the **Adam** optimizer) to push all three of these toward zero simultaneously. Note that this loss does **not** directly use the analytical solution $\cos(\omega t)$ anywhere — the network is being shaped entirely by the physics constraints, not by example answers. The `"l2 relative error"` metric we add is purely for us to *watch* progress against the true solution; it is not part of the optimizer's loss and does not influence training.

In [ ]:
# Fully-connected feedforward network ("FNN"):
#   layer sizes [1, 32, 32, 32, 1], tanh activation, Glorot ("Xavier") weight initialization.
# Glorot initialization picks starting weights with a variance scaled to the layer size,
# which helps gradients neither explode nor vanish early in training.
net = dde.nn.FNN(
    [1, 32, 32, 32, 1],
    "tanh",
    "Glorot uniform",
)

# Combine the physics problem (`data`) with the network (`net`) into a trainable Model.
model = dde.Model(data, net)

# Configure the optimizer before training:
#   - "adam": a popular gradient-descent variant with adaptive per-parameter learning rates.
#   - lr=1e-3: the learning rate (step size) Adam uses to update weights each iteration.
#   - metrics=["l2 relative error"]: report this accuracy metric against `true_solution`
#     during training, purely for our own monitoring (it is NOT part of the training loss).
model.compile(
    optimizer="adam",
    lr=1e-3,
    metrics=["l2 relative error"],
)

# Train for 2000 iterations (gradient-descent steps), printing the current losses
# and metrics every 500 iterations so we can watch training progress.
# `losshistory` records the loss values over time; `train_state` holds the final
# model state (including the best-performing weights seen during training).
losshistory, train_state = model.train(
    iterations=3000,
    display_every=500,
)

Compiling model...
'compile' took 0.678842 s

Training model...

Step      Train loss                        Test loss                         Test metric   
0         [7.46e+00, 1.00e+00, 2.92e-01]    [7.55e+00, 1.00e+00, 2.92e-01]    [1.28e+00]    
500       [2.34e-01, 1.78e-01, 2.57e-06]    [2.36e-01, 1.78e-01, 2.57e-06]    [9.14e-01]    


## Step 4: Evaluate and Plot Results

Training is done — now let's see how well the network actually learned the physics.

The plan:
1. Build a fine grid of 500 time points between $0$ and $5$ (much denser than the 200 points used during training, so we get a smooth curve).
2. Ask the trained model to **predict** $x(t)$ at each of those points — this is a single forward pass through the network, with no further training.
3. Compute the **exact** analytical solution $\cos(\omega t)$ at the same points, using the formula we defined earlier (`true_solution`), purely for comparison.
4. Plot both curves on the same axes. If training went well, the dashed PINN prediction should lie almost on top of the solid exact curve.

In [ ]:
# Build a dense, evenly spaced grid of 500 time points in [0, 5].
# `.reshape(-1, 1)` turns it into a column vector (shape (500, 1)), which is the
# input shape DeepXDE/PyTorch expects: one row per sample, one column per input feature.
t_predict = np.linspace(0, 5, 500).reshape(-1, 1)

# Ask the trained network for its prediction at each of these 500 time points.
# This is pure inference (a forward pass) -- no further training happens here.
x_predict = model.predict(t_predict)

# Evaluate the known exact formula at the same points, for comparison.
x_exact = true_solution(t_predict)

# Flatten the (500, 1) column vectors down to plain 1D arrays of length 500,
# which is the shape matplotlib's plot() expects.
t_plot = t_predict.flatten()
x_pred_plot = x_predict.flatten()
x_exact_plot = x_exact.flatten()

# Plot both curves together to visually compare the PINN's prediction
# against the true analytical solution.
plt.figure(figsize=(10, 5))
plt.plot(t_plot, x_exact_plot,
         label="Exact Analytical",
         linewidth=2)

plt.plot(t_plot, x_pred_plot,
         "--",
         label="PINN Prediction",
         linewidth=2)

plt.title("Simple Harmonic Motion using PINNs")
plt.xlabel("Time")
plt.ylabel("Displacement")
plt.legend()
plt.grid(True)
plt.show()

# If the two curves overlap closely, the network has successfully learned a function
# that satisfies both the ODE and the initial conditions -- without ever being told
# the formula cos(omega * t) directly.